# AnyMatch — Inference on real AllianceChicago candidate pairs

Joins the blocking output (`candidate_pairs_*.parquet`, PATID_A / PATID_B only) with the cleaned MDM population (`MDM_Population_cleaned_v1.csv`), filters to pairs where **both** records have `valid_record = True`, optionally subsamples, and scores the result with the **fine-tuned** GPT-2 checkpoint (`anymatch_alliance_ft_v2`) — the same model evaluated in `anymatch_synthetic_inference.ipynb`.

**Where to run**: locally, from the `AnyMatch/` folder. Launch Jupyter / VS Code with cwd = the `AnyMatch/` directory so relative paths resolve.

**Inference path**: scores **inline** (not via `predict_alliance.py`). The fine-tuned checkpoint ships only `tokenizer.json` (HF *fast* format), which the slow `GPT2Tokenizer` the CLI uses can't load; we load `GPT2TokenizerFast` here (falling back to base `gpt2` BPE) and reuse the repo's exact `df_serializer` (mode4) + `GPTDataset` + an inline `predict`, so serialization is identical to training/serving.

**Hardware**: inference auto-selects **CUDA → Apple-Silicon MPS → CPU**. On an M-series Mac MPS is ~2-3x faster than CPU; pure CPU is ~270 ms/pair. At that rate the full 212k candidate set is several hours locally — move to Colab Pro GPU for the full run (run cells 1–4 then this inference cell in the same kernel).

**PHI reminder**: this notebook reads real patient data. Run only on a machine that satisfies your institution's policy.

## 1. Config

The only cell you normally edit. `N_PAIRS = -1` to score every valid pair; any positive int subsamples the first N (deterministic). `FEATURE_COLS` are positional — `_l` and `_r` must align column-for-column, which is automatic since both sides are joined from the same source.

In [ ]:
import os
from utils.alliance_schema import (
    CANONICAL_RENAMES as FEATURE_RENAMES,
    FEATURE_COLS_SRC, FEATURE_COLS,
)

# === EDIT THESE IF NEEDED ===
# Point at the FINE-TUNED checkpoint (sequential FT on the synthetic corpus) —
# the same one scored in anymatch_synthetic_inference.ipynb. For an A/B against
# the zero-shot base, set CKPT_DIR = 'saved_models/anymatch_all9_gpt2_mode4'.
CKPT_DIR     = 'saved_models/anymatch_alliance_ft_v2'
PAIRS_PATH   = 'data/alliance/candidate_pairs_v1_20260521_035346.parquet'
RECORDS_PATH = 'data/alliance/MDM_Population_cleaned_v1.csv'
OUTPUT_DIR   = 'data/alliance'
OUTPUT_STEM  = 'anymatch_predictions'   # final name: {OUTPUT_STEM}_{MODEL_TAG}_{YYYYMMDD_HHMMSS}.csv

N_PAIRS = 100   # set to -1 to score every valid pair
# ============================
#
# The feature schema (technical MDM column -> clean English attribute name) lives
# in utils/alliance_schema.py so the fine-tune training and BOTH inference
# notebooks stay in lockstep. mode4 emits "<attr>: <value>" per attribute, where
# <attr> is the friendly name; dict order is positional. Full spec §2 schema:
# three separate name fields (first/middle/last — to teach cross-field token
# movement), suffix, address2, ssn + ssn4, email.

MODEL_TAG = os.path.basename(CKPT_DIR)   # used in the output filename

assert os.path.exists('loo.py'), (
    f'Expected to run from the AnyMatch/ directory (cwd={os.getcwd()!r}). '
    "cd into AnyMatch/ before launching Jupyter, or os.chdir() here."
)

# Checkpoint validation. The fine-tuned checkpoint ships the HF *fast* tokenizer
# (tokenizer.json) and safetensors weights; the zero-shot base ships the slow
# tokenizer pair (vocab.json + merges.txt) and may use pytorch_model.bin. Accept
# either layout so this notebook works with both checkpoints unchanged.
if not os.path.exists(os.path.join(CKPT_DIR, 'config.json')):
    raise FileNotFoundError(f'Missing config.json from {CKPT_DIR}. Download the checkpoint folder.')
has_weights = any(os.path.exists(os.path.join(CKPT_DIR, f))
                  for f in ('model.safetensors', 'pytorch_model.bin'))
has_tok = os.path.exists(os.path.join(CKPT_DIR, 'tokenizer.json')) or all(
    os.path.exists(os.path.join(CKPT_DIR, f)) for f in ('vocab.json', 'merges.txt'))
missing = []
if not has_weights:
    missing.append('model weights (model.safetensors / pytorch_model.bin)')
if not has_tok:
    missing.append('tokenizer (tokenizer.json, or vocab.json + merges.txt)')
if missing:
    raise FileNotFoundError(
        f'Missing from {CKPT_DIR}: {missing}. '
        'Download the trained checkpoint folder from Drive and place it there.'
    )
for p in [PAIRS_PATH, RECORDS_PATH]:
    assert os.path.exists(p), f'Missing input: {p}'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Config OK. Scoring with checkpoint: {CKPT_DIR}')

## 2. Load records, filter to `valid_record = True`

Indexing by PATID makes the join in cell 4 a single `.loc[...]` lookup per side.

In [ ]:
import pandas as pd
from utils.alliance_schema import id_str_dtypes, prep_record_df

# Force-read numeric-looking ID columns as strings. Without this, pandas infers
# float64 (because of NaN), prints "358467965.0" and drops leading zeros — which
# wrecks exact-match signal. id_str_dtypes() picks the ID columns present.
_header = pd.read_csv(RECORDS_PATH, nrows=0).columns
records = pd.read_csv(RECORDS_PATH, low_memory=False, dtype=id_str_dtypes(_header))
records_valid = records[records['valid_record']].copy()
print(f'{len(records):,} total records → {len(records_valid):,} with valid_record=True')

missing_cols = [c for c in FEATURE_COLS_SRC if c not in records_valid.columns]
if missing_cols:
    raise KeyError(f'FEATURE_COLS_SRC not found in records: {missing_cols}')

# Rename technical -> friendly and apply set/date transforms ONCE (per-record),
# then index by PATID so the per-side join in cell 8 is an O(1) .loc lookup.
records_valid = prep_record_df(records_valid)
records_valid = records_valid.set_index('PATID')
if not records_valid.index.is_unique:
    dup = records_valid.index[records_valid.index.duplicated()].unique()[:5]
    raise ValueError(f'PATID is not unique in MDM file. Examples: {list(dup)}')

# Sanity: ID columns should be strings now (or NaN), no trailing ".0".
for col in ['ssn', 'ssn4', 'zip']:
    if col in records_valid.columns:
        sample = records_valid[col].dropna().head(3).tolist()
        print(f'  {col}: dtype={records_valid[col].dtype}, sample={sample}')

## 3. Load pairs, keep only pairs where both sides are valid, subsample

In [ ]:
pairs = pd.read_parquet(PAIRS_PATH)[['PATID_A', 'PATID_B']]
pairs = pairs.sample(frac=1, random_state=42)
print(f'{len(pairs):,} candidate pairs from blocking')

valid_ids = set(records_valid.index)
pairs = pairs[pairs['PATID_A'].isin(valid_ids) & pairs['PATID_B'].isin(valid_ids)].reset_index(drop=True)
print(f'{len(pairs):,} pairs where both sides have valid_record=True')

if N_PAIRS != -1:
    pairs = pairs.head(N_PAIRS)
    print(f'Subsampled to first {len(pairs):,} pairs (N_PAIRS={N_PAIRS})')

## 4. Build the `_l` / `_r` wide CSV that `predict_alliance.py` expects

The technical→friendly rename and the per-record transforms already happened in cell 4 via `prep_record_df` (from `utils/alliance_schema`):

- The three name fields (`FirstNM_clean`/`MiddleNM_clean`/`LastNM_clean` → `first_name`/`middle_name`/`last_name`), plus `suffix` and `address2`, ride through as raw clean strings — each its own attribute, so the model can learn cross-field name-token movement (spec §2).
- `Phones_set` → sorted, space-joined string (so the `{'3125551234'}` set literal doesn't pollute the prompt).
- `dob` → normalized `YYYY-MM-DD`.

This cell only joins each side by PATID and writes the wide CSV.

In [ ]:
# Build the _l / _r wide table. The per-record rename + set/date transforms
# already ran in cell 4 (prep_record_df), so here we just join each side by PATID.
# FEATURE_COLS is the friendly, positional schema (first_name, middle_name,
# last_name, suffix, dob, ssn, ssn4, sex, address, address2, ...).
wide = pairs.reset_index(drop=True).copy()
for side, patid_col in [('l', 'PATID_A'), ('r', 'PATID_B')]:
    side_df = (
        records_valid.loc[pairs[patid_col].values, FEATURE_COLS]
        .reset_index(drop=True)
        .add_suffix(f'_{side}')
    )
    wide = pd.concat([wide, side_df], axis=1)

# Sanity check — DOB and last_name columns should be populated where source was.
for col in ['dob_l', 'dob_r', 'last_name_l', 'last_name_r']:
    n_pop = wide[col].astype('string').str.len().gt(0).sum()
    print(f'{col}: {n_pop}/{len(wide)} populated')

intermediate_csv = f'{OUTPUT_DIR}/_anymatch_input_temp.csv'
wide.to_csv(intermediate_csv, index=False)
print(f'Wrote {len(wide):,} rows × {len(wide.columns)} cols to {intermediate_csv}')
wide.head(3)

## 5. Peek at the model input

Before running inference, build the exact prompt strings that AnyMatch will see for the first few pairs. Useful for sanity-checking that columns line up, missing values are showing as `N/A` (not literal `nan` strings), and the token count stays under GPT-2's 1024-token cap.

In [ ]:
# Build the exact prompt that AnyMatch will see for each pair.
# df_serializer (mode4) emits one string per row using the template:
#   "Given the attributes of two records, are they the same?
#    Record A is name: <v>, dob: <v>, ssn: <v>, ....
#    Record B is name: <v>, dob: <v>, ssn: <v>, ...."
# The attribute names ("name", "dob", "ssn", ...) come from the dataframe column
# names (sans _l/_r) — which is why we renamed in cell 4. Missing values become
# the literal token "N/A" via .fillna('N/A').
# PATID columns are bystanders — they ride through the CSV but are NOT in the prompt.
from utils.data_utils import df_serializer

preview = wide.copy()
if 'label' not in preview.columns:
    preview['label'] = 0   # df_serializer requires the column; placeholder only

serialized = df_serializer(preview, 'mode4')
N_PREVIEW = min(3, len(serialized))
for i in range(N_PREVIEW):
    print(f'=== Pair {i}  PATID_A={wide["PATID_A"].iloc[i]}  PATID_B={wide["PATID_B"].iloc[i]} ===')
    print(serialized['text'].iloc[i])
    print()

# Quick token-budget check (GPT-2 hard limit is 1024 tokens per sequence).
# from transformers import GPT2Tokenizer
# _tok = GPT2Tokenizer.from_pretrained(CKPT_DIR)
# _lens = serialized['text'].apply(lambda t: len(_tok.encode(t)))
# print(f'Token lengths — min={_lens.min()}, mean={_lens.mean():.0f}, '
#       f'max={_lens.max()} (GPT-2 cap = 1024)')

## 6. Run inference (inline)

Scores every pair **in-process**, reusing the repo's own serialization and prediction code so the prompt is byte-identical to training — mirroring `anymatch_synthetic_inference.ipynb`. We do **not** shell out to `predict_alliance.py` here: the fine-tuned checkpoint ships only `tokenizer.json` (HF *fast* format), which the slow `GPT2Tokenizer` the CLI uses can't load.

- `df_serializer(df, 'mode4')` → `Given the attributes of two records, are they the same? Record A is first_name: <v>, ... . Record B is ... .`
- `GPTDataset` tokenizes; a small inline scorer returns `pred` (argmax 0/1) and `match_prob` = `softmax(logits)[:, 1]`. It mirrors `utils.train_eval.predict` but selects the device **CUDA → Apple-Silicon MPS → CPU** (the shared `predict()` only checks CUDA), with an automatic CPU fallback if an op isn't MPS-supported.
- **Tokenizer**: tries the checkpoint's fast tokenizer, falling back to the base `gpt2` BPE — fine-tuning never changes GPT-2's vocab, so token IDs are identical either way.

`PATID_A` / `PATID_B` and all other bystander columns ride through to the output CSV untouched.

In [ ]:
import os, time
from datetime import datetime
import torch
from torch.utils.data import DataLoader
from transformers import GPT2Tokenizer, GPT2TokenizerFast, GPT2ForSequenceClassification

from utils.data_utils import df_serializer
from data import GPTDataset

# Let any op MPS doesn't implement fall back to CPU instead of erroring.
os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')

BATCH_SIZE    = 32
MAX_LEN       = 1024    # GPT-2 context cap; mode4 patient prompts are ~150-250 tok, so nothing is filtered
TOKENIZER_SRC = 'gpt2'  # GPT-2 BPE is fixed; fine-tuning never changes the vocab

ts = datetime.now().strftime('%Y%m%d_%H%M%S')
output_csv = f'{OUTPUT_DIR}/{OUTPUT_STEM}_{MODEL_TAG}_{ts}.csv'

# Serialize from the IN-MEMORY `wide` (built in cell 8) so the prompt matches
# training exactly — re-reading the temp CSV would let pandas re-infer float64
# for zip/ssn4/phone ("60640.0"), diverging from how training serialized them.
serial_src = wide.copy()
if 'label' not in serial_src.columns:
    serial_src['label'] = 0   # df_serializer requires the column; placeholder only
serialized = df_serializer(serial_src.copy(), 'mode4')
print(f'Serializing {len(wide)} pairs (in memory)')
print('Example serialized prompt:\n', serialized['text'].iloc[0][:400], '...\n')

# Tokenizer: try the checkpoint's own tokenizer.json (fast), fall back to base
# gpt2 BPE if the local `tokenizers` lib is older than the one that wrote it
# ("data did not match any variant ... ModelWrapper"). Vocab is identical, so
# token IDs match training either way.
try:
    tokenizer = GPT2TokenizerFast.from_pretrained(CKPT_DIR)
    print('Tokenizer: fast, from checkpoint')
except Exception as e:
    print(f'[info] fast tokenizer from checkpoint failed ({type(e).__name__}); '
          f'falling back to base {TOKENIZER_SRC!r} tokenizer (identical BPE).')
    tokenizer = GPT2Tokenizer.from_pretrained(TOKENIZER_SRC)

model = GPT2ForSequenceClassification.from_pretrained(CKPT_DIR)
model.config.pad_token_id = model.config.eos_token_id

dataset = GPTDataset(tokenizer, serialized, max_len=MAX_LEN)
assert len(dataset) == len(wide), (
    f'{len(wide) - len(dataset)} row(s) dropped by the length filter — raise MAX_LEN.')


def pick_device():
    """Prefer CUDA, then Apple-Silicon MPS, else CPU."""
    if torch.cuda.is_available():
        return torch.device('cuda')
    mps = getattr(torch.backends, 'mps', None)
    if mps is not None and mps.is_available() and mps.is_built():
        return torch.device('mps')
    return torch.device('cpu')


@torch.no_grad()
def score(model, dataset, batch_size, device):
    """Inline equivalent of utils.train_eval.predict, but device-selectable
    (the shared predict() only checks CUDA). Returns (preds, probs)."""
    dl = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=dataset.collate_fn)
    model.to(device); model.eval()
    preds, probs = [], []
    for batch in dl:
        batch = {k: v.to(device) for k, v in batch.items()}
        logits = model(**batch).logits.detach().float()
        probs += torch.softmax(logits, dim=-1)[:, 1].cpu().numpy().tolist()
        preds += logits.argmax(dim=-1).cpu().numpy().flatten().tolist()
    return preds, probs


device = pick_device()
print(f'Scoring {len(dataset)} pairs on {device.type.upper()} (batch={BATCH_SIZE}) ...')
t0 = time.perf_counter()
try:
    preds, probs = score(model, dataset, BATCH_SIZE, device)
except (RuntimeError, NotImplementedError) as e:
    if device.type == 'mps':
        print(f'[warn] MPS scoring failed ({type(e).__name__}: {str(e)[:120]}); retrying on CPU.')
        device = torch.device('cpu')
        preds, probs = score(model, dataset, BATCH_SIZE, device)
    else:
        raise
elapsed = time.perf_counter() - t0

# Write the wide table + predictions. PATID_A / PATID_B and all bystander
# columns ride through untouched.
out = wide.copy()
out['pred'] = preds
out['match_prob'] = probs
out.to_csv(output_csv, index=False)

print(f'\n{len(out):,} pairs in {elapsed:.1f}s ({elapsed/max(len(out),1)*1000:.0f} ms/pair)')
print(f'Predicted-match rate : {out["pred"].mean()*100:.1f}%')
print(f'Output: {output_csv}')

## 7. Inspect predictions

No ground-truth labels here, so no F1. Useful diagnostics:
- pred distribution
- `match_prob` histogram
- top-10 most confident matches (sanity check that obvious matches score high)
- 10 pairs closest to 0.5 (threshold-calibration candidates / hand-label targets)

In [ ]:
out = pd.read_csv(output_csv)
print('--- pred distribution ---')
print(out['pred'].value_counts())
print('\n--- match_prob summary ---')
display(out['match_prob'].describe())
out['match_prob'].plot.hist(bins=50, title='match_prob distribution');

In [ ]:
alt_cols = [c for col in FEATURE_COLS for c in (f'{col}_l', f'{col}_r')]
view_cols = ['match_prob', 'PATID_A', 'PATID_B', *alt_cols]

with pd.option_context('display.max_columns', None, 'display.width', None):
    print('--- Top-10 most confident matches ---')
    display(out.nlargest(10, 'match_prob')[view_cols].round({'match_prob': 3}))

    print('--- 10 pairs closest to 0.5 (uncertain) ---')
    near_idx = (out['match_prob'] - 0.5).abs().sort_values().index[:10]
    display(out.loc[near_idx, view_cols].round({'match_prob': 3}))

## 8. Notes

- **Threshold tuning**: `pred` uses 0.5 by default. On the synthetic eval the fine-tuned model was near-perfect-recall but produced a handful of false merges, so for production sweep `match_prob` against a hand-labeled holdout (see the threshold sweep in `anymatch_synthetic_inference.ipynb`) and lean toward a high-precision operating point (e.g. P≥0.99) for auto-merge, routing the mid band to human review.
- **Scaling**: at ~270 ms/pair on Mac CPU (faster on MPS), the full 212k candidate set is several hours locally. For the full run, set `N_PAIRS = -1` and run cells 1–4 + the inference cell on a Colab Pro GPU in one kernel.
- **Schema invariant**: column order in `FEATURE_COLS` is positional. If you add or reorder columns, both sides re-emit symmetrically — no separate L/R definition to keep in sync.
- **A/B**: to compare against the zero-shot base, set `CKPT_DIR = 'saved_models/anymatch_all9_gpt2_mode4'` in cell 1 and re-run — output files are named per-checkpoint (`{OUTPUT_STEM}_{MODEL_TAG}_...`) so they won't clobber each other.
- The temp file `_anymatch_input_temp.csv` (cell 8) is no longer consumed by inference — scoring reads the in-memory `wide` frame — but is left on disk for debugging; safe to delete.